# CMV Calibration Round — Inter-Rater Agreement

Compares the three calibration files, `data/human/calibration_H{1,2,3}.csv` (5 OPs × 15 pairs = 75 pairs each).

**Key step:** each annotator saw a different A/B orientation (the orientation hash includes the annotator ID),
so all judgments are re-expressed relative to a canonical pair orientation taken from `pair_key = post|R1|R2`
(R1 < R2 lexicographically). Agreement is never computed on the raw "A"/"B" choice labels.

Reliability is reported primarily with Krippendorff's α (thesis §3.4), with supporting Fleiss/Cohen κ,
raw agreement, and Bradley–Terry → Kendall τ_b rank correspondence (thesis §3.2).
Behavioral outcome columns are never loaded.

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
import sys
from pathlib import Path

# Repository root: the first folder at or above the working directory that contains data/annotation_input.csv.
ROOT      = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "annotation_input.csv").exists())
ANNO_DIR  = ROOT / "data" / "human"
INPUT_CSV = ROOT / "data" / "annotation_input.csv"   # used only for display text
OUT_DIR   = ROOT / "results" / "calibration"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Rater label -> file. Relative names are resolved inside ANNO_DIR.
RATER_FILES = {
    "H1": "calibration_H1.csv",
    "H2": "calibration_H2.csv",
    "H3": "calibration_H3.csv",
}

N_BOOT = 1000      # bootstrap replicates
SEED   = 20260924
BT_LAMBDA = 0.1    # Bradley–Terry ridge penalty

print("Calibration files:", sorted(p.name for p in ANNO_DIR.glob("calibration_*.csv")))


In [ ]:
# ============================================================
# IMPORTS + GENERIC HELPERS
# ============================================================
import itertools, warnings
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize
from sklearn.metrics import cohen_kappa_score
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)
warnings.filterwarnings("ignore", category=RuntimeWarning)

rng = np.random.default_rng(SEED)

def as_bool(v):
    if pd.isna(v):
        return False
    if isinstance(v, str):
        return v.strip().lower() in ("true", "1", "yes")
    return bool(v)

def resolve_path(p):
    p = Path(p)
    return p if p.is_absolute() else ANNO_DIR / p

def op_of(unit_key):
    """pair_key and pair-side keys both start with post_id."""
    return str(unit_key).split("|")[0]

In [ ]:
# ============================================================
# LOAD + VALIDATE
# ============================================================
REQUIRED = [
    "pair_key", "annotator_id", "instruction_version", "post_id", "sample_post_order", "pair_in_post",
    "response_id_a", "response_id_b", "ab_flipped", "task_order",
    "cogency_choice", "cogency_winner_response_id", "cogency_signed",
    "effect_choice", "effect_winner_response_id", "effect_signed",
    "evidence_a_present", "evidence_b_present", "evidence_presence_pattern",
    "evidence_choice", "evidence_winner_response_id", "evidence_signed",
    "a_nonargumentative", "b_nonargumentative", "a_unreadable", "b_unreadable",
    "seconds_on_pair_total", "revision_count",
]
STR_COLS = ["pair_key", "post_id", "response_id_a", "response_id_b",
            "cogency_winner_response_id", "effect_winner_response_id", "evidence_winner_response_id"]

raw = {}
for rater, f in RATER_FILES.items():
    path = resolve_path(f)
    assert path.exists(), f"[{rater}] file not found: {path}"
    df = pd.read_csv(path, dtype={c: str for c in STR_COLS}, encoding="utf-8-sig")
    missing = [c for c in REQUIRED if c not in df.columns]
    assert not missing, f"[{rater}] missing columns: {missing}"
    assert len(df) == 75, f"[{rater}] expected 75 rows, found {len(df)}"
    assert df["pair_key"].is_unique, f"[{rater}] duplicate pair_key rows"
    raw[rater] = df
    print(f"[{rater}] {path.name}: {len(df)} rows | annotator_id in file = {df['annotator_id'].unique().tolist()} "
          f"| version = {df['instruction_version'].unique().tolist()} | task_order = {df['task_order'].unique().tolist()}")

RATERS = list(raw)
assert len(RATERS) >= 2

key_sets = {r: set(d["pair_key"]) for r, d in raw.items()}
common = set.intersection(*key_sets.values())
for r, s in key_sets.items():
    assert s == common, f"[{r}] pair_key set differs from other raters ({len(s - common)} extra, {len(common - s)} missing)"
print(f"\nAll {len(RATERS)} files cover the same {len(common)} pairs across "
      f"{len({op_of(k) for k in common})} OPs.")

# Display text only (no behavioral columns are read).
TEXT, OP_TITLE = {}, {}
if INPUT_CSV.exists():
    _txt = pd.read_csv(INPUT_CSV, usecols=["post_id", "response_id", "op_title", "response_text_clean"],
                       dtype={"post_id": str, "response_id": str}, encoding="utf-8-sig")
    TEXT = dict(zip(_txt["response_id"], _txt["response_text_clean"].fillna("").astype(str)))
    OP_TITLE = _txt.drop_duplicates("post_id").set_index("post_id")["op_title"].to_dict()
    del _txt
    print("Loaded response text for the disagreement sheet (safe columns only).")
else:
    print("Input CSV not found — disagreement sheet will omit text.")

In [ ]:
# ============================================================
# CANONICAL ORIENTATION
#   R1 = lexicographically smaller response id, R2 = larger (from pair_key)
#   *_r2wins = 1 if R2 chosen; *_signed_c in {-2,-1,+1,+2}, positive = R2
#   *_rank4  in {1,2,3,4} = Strong R1, Weak R1, Weak R2, Strong R2
# ============================================================
SIGNED = {"Strong A": -2, "Weak A": -1, "Weak B": 1, "Strong B": 2}
RANK4  = {-2: 1, -1: 2, 1: 3, 2: 4}
LABEL4 = {1: "Strong R1", 2: "Weak R1", 3: "Weak R2", 4: "Strong R2"}
PATTERN_CODE = {"neither": 0, "R1_only": 1, "R2_only": 2, "both": 3}

def canonicalize(df, rater):
    d = df.reset_index(drop=True).copy()
    kp = d["pair_key"].str.split("|", expand=True)
    assert kp.shape[1] == 3, f"[{rater}] unexpected pair_key format"
    d["R1"], d["R2"] = kp[1], kp[2]
    assert (d["post_id"] == kp[0]).all(), f"[{rater}] post_id does not match pair_key"

    ok = (((d.response_id_a == d.R1) & (d.response_id_b == d.R2)) |
          ((d.response_id_a == d.R2) & (d.response_id_b == d.R1)))
    assert ok.all(), f"[{rater}] response_id_a/b do not match pair_key"
    a_is_r1 = (d.response_id_a == d.R1).to_numpy()
    orient = np.where(a_is_r1, 1, -1)

    out = d[["pair_key", "post_id", "sample_post_order", "R1", "R2"]].copy()
    out["rater"] = rater
    out["a_is_r1"] = a_is_r1

    for k in ("cogency", "effect"):
        choice = d[f"{k}_choice"]
        assert choice.isin(list(SIGNED)).all(), f"[{rater}] invalid {k}_choice values"
        s_raw = choice.map(SIGNED).astype(int).to_numpy()
        stored = pd.to_numeric(d[f"{k}_signed"], errors="coerce").to_numpy()
        assert (stored == s_raw).all(), f"[{rater}] {k}_signed inconsistent with {k}_choice"
        s = s_raw * orient
        r2wins = (s > 0).astype(int)
        winner_is_r2 = (d[f"{k}_winner_response_id"] == d.R2).astype(int).to_numpy()
        assert (winner_is_r2 == r2wins).all(), f"[{rater}] {k} winner id inconsistent with choice/orientation"
        out[f"{k}_r2wins"] = r2wins
        out[f"{k}_signed_c"] = s
        out[f"{k}_rank4"] = pd.Series(s).map(RANK4).to_numpy()
        out[f"{k}_strong"] = (np.abs(s) == 2).astype(int)
        out[f"{k}_choseA"] = choice.str.endswith("A").astype(int).to_numpy()

    eva = d["evidence_a_present"].map(as_bool).to_numpy()
    evb = d["evidence_b_present"].map(as_bool).to_numpy()
    out["ev_r1"] = np.where(a_is_r1, eva, evb).astype(int)
    out["ev_r2"] = np.where(a_is_r1, evb, eva).astype(int)
    both = ((out.ev_r1 == 1) & (out.ev_r2 == 1)).to_numpy()
    pattern = np.select([both, out.ev_r1.to_numpy() == 1, out.ev_r2.to_numpy() == 1],
                        ["both", "R1_only", "R2_only"], "neither")
    out["ev_pattern"] = pattern
    out["ev_pattern_code"] = pd.Series(pattern).map(PATTERN_CODE).to_numpy()

    ev_s = d["evidence_choice"].map(SIGNED).to_numpy(dtype=float) * orient
    n_bad = int((both & np.isnan(ev_s)).sum())
    if n_bad:
        print(f"WARNING [{rater}]: {n_bad} pairs have both evidence boxes checked but no evidence comparison.")
    ev_s[~both] = np.nan
    out["evidence_signed_c"] = ev_s
    out["evidence_r2wins"] = np.where(np.isnan(ev_s), np.nan, (ev_s > 0).astype(float))
    out["evidence_rank4"] = pd.Series(ev_s).map(RANK4).to_numpy()

    for flag, a_col, b_col in (("nonarg", "a_nonargumentative", "b_nonargumentative"),
                               ("unread", "a_unreadable", "b_unreadable")):
        fa = d[a_col].map(as_bool).to_numpy(); fb = d[b_col].map(as_bool).to_numpy()
        out[f"{flag}_r1"] = np.where(a_is_r1, fa, fb).astype(int)
        out[f"{flag}_r2"] = np.where(a_is_r1, fb, fa).astype(int)

    out["ce_discordant"] = (out.cogency_r2wins != out.effect_r2wins).astype(int)   # Z in Eq. 11
    out["seconds"] = pd.to_numeric(d["seconds_on_pair_total"], errors="coerce").to_numpy()
    out["revision_count"] = pd.to_numeric(d["revision_count"], errors="coerce").to_numpy()
    return out

long = pd.concat([canonicalize(raw[r], r) for r in RATERS], ignore_index=True)
OP_IDS = (long[["post_id", "sample_post_order"]].drop_duplicates()
          .sort_values("sample_post_order")["post_id"].tolist())

def wide(var, data=None):
    data = long if data is None else data
    return data.pivot(index="pair_key", columns="rater", values=var)[RATERS]

# Response-in-pair level table (each pair contributes an R1 unit and an R2 unit)
side_rows = []
for side in ("r1", "r2"):
    t = long[["pair_key", "rater", f"ev_{side}", f"nonarg_{side}", f"unread_{side}"]].copy()
    t.columns = ["pair_key", "rater", "ev_present", "nonarg", "unread"]
    t["unit"] = t["pair_key"] + "|" + side
    side_rows.append(t)
long_side = pd.concat(side_rows, ignore_index=True)

def wide_side(var):
    return long_side.pivot(index="unit", columns="rater", values=var)[RATERS]

print("Canonicalization passed all consistency checks.")
print(f"A/B orientation differs from R1/R2 on: " +
      ", ".join(f"{r}={(~long[long.rater == r].a_is_r1).mean():.0%}" for r in RATERS))

In [ ]:
# ============================================================
# AGREEMENT STATISTICS
# ============================================================
def kripp_alpha(X, level="nominal"):
    """Krippendorff's alpha. X: units x raters array with NaN for missing."""
    X = np.asarray(X, dtype=float)
    present = X[~np.isnan(X)]
    vals = np.unique(present)
    if len(vals) < 2:
        return np.nan
    V = len(vals)
    idx = {v: i for i, v in enumerate(vals)}
    o = np.zeros((V, V))
    for row in X:
        r = row[~np.isnan(row)]
        m = len(r)
        if m < 2:
            continue
        cnt = np.zeros(V)
        for v in r:
            cnt[idx[v]] += 1
        o += (np.outer(cnt, cnt) - np.diag(cnt)) / (m - 1)
    n_c = o.sum(axis=1)
    n = n_c.sum()
    if n <= 1:
        return np.nan
    if level == "nominal":
        d2 = 1.0 - np.eye(V)
    elif level == "interval":
        d2 = (vals[:, None] - vals[None, :]) ** 2
    elif level == "ordinal":
        d2 = np.zeros((V, V))
        for c in range(V):
            for k in range(V):
                a, b = min(c, k), max(c, k)
                d2[c, k] = (n_c[a:b + 1].sum() - (n_c[c] + n_c[k]) / 2.0) ** 2
    else:
        raise ValueError(level)
    Do = (o * d2).sum() / n
    De = (np.outer(n_c, n_c) * d2).sum() / (n * (n - 1))
    return 1.0 - Do / De if De > 0 else np.nan

def fleiss_kappa(W):
    X = W.dropna().to_numpy()
    if len(X) == 0:
        return np.nan
    n = X.shape[1]
    cats = np.unique(X)
    if len(cats) < 2:
        return np.nan
    counts = np.stack([(X == c).sum(axis=1) for c in cats], axis=1)
    P_i = ((counts ** 2).sum(axis=1) - n) / (n * (n - 1))
    p_j = counts.sum(axis=0) / counts.sum()
    Pe = (p_j ** 2).sum()
    return (P_i.mean() - Pe) / (1 - Pe) if Pe < 1 else np.nan

def pct_pairwise(W):
    vals = []
    for a, b in itertools.combinations(W.columns, 2):
        m = W[[a, b]].dropna()
        if len(m):
            vals.append((m[a] == m[b]).mean())
    return float(np.mean(vals)) if vals else np.nan

def pct_unanimous(W):
    m = W.dropna()
    return float((m.nunique(axis=1) == 1).mean()) if len(m) else np.nan

def kappa(a, b, weights=None, labels=None):
    m = pd.concat([a, b], axis=1).dropna()
    if len(m) < 2:
        return np.nan
    try:
        return cohen_kappa_score(m.iloc[:, 0], m.iloc[:, 1], weights=weights, labels=labels)
    except Exception:
        return np.nan

def boot_ci(W, fn, B=N_BOOT, cluster=False):
    """Percentile CI. cluster=True resamples OPs (thesis' resampling unit) instead of units."""
    vals = []
    if not cluster:
        n = len(W)
        for _ in range(B):
            vals.append(fn(W.iloc[rng.integers(0, n, n)]))
    else:
        groups = pd.Series(W.index.map(op_of)).groupby(lambda i: op_of(W.index[i])).indices
        keys = list(groups)
        for _ in range(B):
            pick = rng.choice(keys, len(keys), replace=True)
            vals.append(fn(W.iloc[np.concatenate([groups[g] for g in pick])]))
    vals = np.asarray(vals, float)
    vals = vals[~np.isnan(vals)]
    return (np.percentile(vals, 2.5), np.percentile(vals, 97.5)) if len(vals) else (np.nan, np.nan)

def n_units(W):
    return int((W.notna().sum(axis=1) >= 2).sum())

print("Agreement functions ready.")

In [ ]:
# ============================================================
# OPTIONAL: VERIFY OWN ALPHA AGAINST THE `krippendorff` PACKAGE
# ============================================================
import subprocess
try:
    import krippendorff
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "krippendorff"], check=False)
    try:
        import krippendorff
    except ImportError:
        krippendorff = None

if krippendorff is None:
    print("Package unavailable — skipping cross-check.")
else:
    for var, lvl in [("cogency_r2wins", "nominal"), ("cogency_rank4", "ordinal"),
                     ("effect_r2wins", "nominal"), ("effect_rank4", "ordinal"),
                     ("evidence_r2wins", "nominal")]:
        W = wide(var)
        mine = kripp_alpha(W.to_numpy(), lvl)
        pkg = krippendorff.alpha(reliability_data=W.to_numpy(dtype=float).T, level_of_measurement=lvl)
        print(f"{var:18s} {lvl:8s}  own={mine: .4f}   package={pkg: .4f}")

In [ ]:
# ============================================================
# HEADLINE RELIABILITY TABLE
# ============================================================
def reliability_row(label, W, level, binary=False):
    fn = lambda x: kripp_alpha(x.to_numpy(), level)
    a = fn(W)
    lo_u, hi_u = boot_ci(W, fn, cluster=False)
    lo_c, hi_c = boot_ci(W, fn, cluster=True)
    return {
        "measure": label, "level": level, "n_units(≥2 raters)": n_units(W),
        "kripp_alpha": a,
        "CI95_unit_boot": f"[{lo_u:.3f}, {hi_u:.3f}]",
        "CI95_OP_cluster_boot": f"[{lo_c:.3f}, {hi_c:.3f}]",
        "fleiss_kappa": fleiss_kappa(W) if binary else np.nan,
        "mean_pairwise_agree": pct_pairwise(W),
        "pct_unanimous": pct_unanimous(W),
    }

rows = []
for k, name in (("cogency", "Cogency"), ("effect", "Expected rhetorical effectiveness")):
    rows.append(reliability_row(f"{name} — winner (binary)", wide(f"{k}_r2wins"), "nominal", binary=True))
    rows.append(reliability_row(f"{name} — 4-point", wide(f"{k}_rank4"), "ordinal"))
    rows.append(reliability_row(f"{name} — 4-point", wide(f"{k}_rank4"), "interval"))
rows.append(reliability_row("Evidence present (per response-in-pair)", wide_side("ev_present"), "nominal", binary=True))
rows.append(reliability_row("Evidence presence pattern (per pair)", wide("ev_pattern_code"), "nominal", binary=True))
rows.append(reliability_row("Evidence comparison — winner (when rater marked both)", wide("evidence_r2wins"), "nominal"))
rows.append(reliability_row("Evidence comparison — 4-point", wide("evidence_rank4"), "ordinal"))
rows.append(reliability_row("Nonargumentative flag (per response-in-pair)", wide_side("nonarg"), "nominal", binary=True))
rows.append(reliability_row("C–E discordance Z (Eq. 11)", wide("ce_discordant"), "nominal", binary=True))

summary = pd.DataFrame(rows)
display(Markdown("### Inter-rater reliability — calibration baseline"))
display(summary.style.format({"kripp_alpha": "{:.3f}", "fleiss_kappa": "{:.3f}",
                              "mean_pairwise_agree": "{:.1%}", "pct_unanimous": "{:.1%}"}, na_rep="—"))

# Strength agreement conditional on direction agreement
cond = []
for k in ("cogency", "effect"):
    D, S = wide(f"{k}_r2wins"), wide(f"{k}_strong")
    same_dir = D.nunique(axis=1) == 1
    cond.append({"construct": k, "pairs_unanimous_on_winner": int(same_dir.sum()),
                 "…of which unanimous on Strong/Weak": float((S[same_dir].nunique(axis=1) == 1).mean()),
                 "pairs with Strong R1 AND Strong R2 present": int(((wide(f"{k}_rank4") == 1).any(axis=1) &
                                                                    (wide(f"{k}_rank4") == 4).any(axis=1)).sum())})
display(Markdown("### Strength agreement and strong reversals"))
display(pd.DataFrame(cond).style.format({"…of which unanimous on Strong/Weak": "{:.1%}"}))

In [ ]:
# ============================================================
# PAIRWISE RATER AGREEMENT
# ============================================================
PAIR_VARS = [
    ("cogency_r2wins", "Cogency winner", None, None),
    ("cogency_rank4", "Cogency 4-pt (quadratic κw)", "quadratic", [1, 2, 3, 4]),
    ("effect_r2wins", "Effectiveness winner", None, None),
    ("effect_rank4", "Effectiveness 4-pt (quadratic κw)", "quadratic", [1, 2, 3, 4]),
    ("ev_pattern_code", "Evidence pattern", None, None),
    ("ce_discordant", "C–E discordance Z", None, None),
]
prow = []
for r1, r2 in itertools.combinations(RATERS, 2):
    for var, label, w, labs in PAIR_VARS:
        W = wide(var)[[r1, r2]].dropna()
        prow.append({"rater_pair": f"{r1} vs {r2}", "measure": label, "n": len(W),
                     "pct_agree": (W[r1] == W[r2]).mean(),
                     "kappa": kappa(W[r1], W[r2], weights=w, labels=labs),
                     "kripp_alpha": kripp_alpha(W.to_numpy(), "ordinal" if w else "nominal")})
    Ws = wide_side("ev_present")[[r1, r2]].dropna()
    prow.append({"rater_pair": f"{r1} vs {r2}", "measure": "Evidence present (response-in-pair)", "n": len(Ws),
                 "pct_agree": (Ws[r1] == Ws[r2]).mean(), "kappa": kappa(Ws[r1], Ws[r2]),
                 "kripp_alpha": kripp_alpha(Ws.to_numpy(), "nominal")})
pairwise = pd.DataFrame(prow)
display(Markdown("### Pairwise agreement"))
display(pairwise.pivot(index="measure", columns="rater_pair", values="kappa").round(3)
        .rename_axis(None, axis=1).style.set_caption("Cohen's κ (weighted where noted)"))
display(pairwise.pivot(index="measure", columns="rater_pair", values="pct_agree")
        .rename_axis(None, axis=1).style.format("{:.1%}").set_caption("Raw % agreement"))

In [ ]:
# ============================================================
# LEAVE-ONE-OUT ALPHA + "MATCHES THE OTHERS WHEN THEY AGREE"
# ============================================================
loo = []
for var, level, label in [("cogency_r2wins", "nominal", "Cogency winner"),
                          ("cogency_rank4", "ordinal", "Cogency 4-pt"),
                          ("effect_r2wins", "nominal", "Effectiveness winner"),
                          ("effect_rank4", "ordinal", "Effectiveness 4-pt")]:
    W = wide(var)
    row = {"measure": label, "alpha_all": kripp_alpha(W.to_numpy(), level)}
    for r in RATERS:
        row[f"alpha_without_{r}"] = kripp_alpha(W.drop(columns=r).to_numpy(), level)
    loo.append(row)
display(Markdown("### Leave-one-out α (a large increase when a rater is dropped flags that rater)"))
display(pd.DataFrame(loo).set_index("measure").round(3))

if len(RATERS) >= 3:
    match = []
    for r in RATERS:
        others = [x for x in RATERS if x != r]
        row = {"rater": r}
        for var, label in [("cogency_r2wins", "Cogency"), ("effect_r2wins", "Effectiveness"),
                           ("ev_pattern_code", "Evidence pattern")]:
            W = wide(var)
            mask = W[others].nunique(axis=1) == 1
            row[f"{label}: n others agree"] = int(mask.sum())
            row[f"{label}: % rater matches"] = (W.loc[mask, r] == W.loc[mask, others[0]]).mean()
        match.append(row)
    display(Markdown("### When the other raters agree with each other, how often does this rater match them?"))
    display(pd.DataFrame(match).set_index("rater").style.format(
        {c: "{:.1%}" for c in pd.DataFrame(match).columns if "%" in c}))

In [ ]:
# ============================================================
# PER-OP AGREEMENT
# ============================================================
oprows = []
for pid in OP_IDS:
    keys = long.loc[long.post_id == pid, "pair_key"].unique()
    row = {"post_id": pid, "op_title": str(OP_TITLE.get(pid, ""))[:70]}
    for k, lab in (("cogency", "C"), ("effect", "E")):
        Wb = wide(f"{k}_r2wins").loc[keys]
        W4 = wide(f"{k}_rank4").loc[keys]
        row[f"{lab}_alpha_bin"] = kripp_alpha(Wb.to_numpy(), "nominal")
        row[f"{lab}_alpha_ord"] = kripp_alpha(W4.to_numpy(), "ordinal")
        row[f"{lab}_unanimous"] = pct_unanimous(Wb)
    row["Ev_pattern_unanimous"] = pct_unanimous(wide("ev_pattern_code").loc[keys])
    oprows.append(row)
per_op = pd.DataFrame(oprows)
display(Markdown("### Per-OP agreement (15 pairs per OP — interpret descriptively)"))
display(per_op.style.format({c: "{:.3f}" for c in per_op.columns if "alpha" in c} |
                            {c: "{:.0%}" for c in per_op.columns if "unanimous" in c}, na_rep="—"))

In [ ]:
# ============================================================
# RATER TENDENCIES (position bias, scale use, flags, time)
# ============================================================
trows = []
for r in RATERS:
    g = long[long.rater == r]
    row = {"rater": r}
    for k, lab in (("cogency", "C"), ("effect", "E")):
        kA = int(g[f"{k}_choseA"].sum())
        row[f"{lab}: % chose A"] = kA / len(g)
        row[f"{lab}: A-bias p"] = stats.binomtest(kA, len(g), 0.5).pvalue
        row[f"{lab}: % Strong"] = g[f"{k}_strong"].mean()
    row["C–E discordant (Z=1)"] = g["ce_discordant"].mean()
    row["% responses w/ evidence"] = (g.ev_r1.sum() + g.ev_r2.sum()) / (2 * len(g))
    row["% pairs both evidence"] = (g.ev_pattern == "both").mean()
    row["% responses nonarg"] = (g.nonarg_r1.sum() + g.nonarg_r2.sum()) / (2 * len(g))
    row["% responses unreadable"] = (g.unread_r1.sum() + g.unread_r2.sum()) / (2 * len(g))
    row["median sec/pair"] = g["seconds"].median()
    row["pairs revised"] = int((g["revision_count"] > 0).sum())
    trows.append(row)
tend = pd.DataFrame(trows).set_index("rater")
pct_cols = [c for c in tend.columns if c.startswith(("C:", "E:", "C–E", "%")) and "p" != c[-1]]
display(Markdown("### Rater tendencies"))
display(tend.style.format({c: "{:.1%}" for c in tend.columns if "%" in c or "discordant" in c} |
                          {c: "{:.3f}" for c in tend.columns if c.endswith("p")} |
                          {"median sec/pair": "{:.0f}"}))

In [ ]:
# ============================================================
# BRADLEY–TERRY WITHIN OP  →  KENDALL τ_b  (thesis §2.4, §3.2)
# ============================================================
def fit_bt(items, comps, lam=BT_LAMBDA):
    ix = {it: i for i, it in enumerate(items)}
    w = np.array([ix[a] for a, b in comps]); l = np.array([ix[b] for a, b in comps])
    def f(th):
        diff = th[w] - th[l]
        nll = np.logaddexp(0, -diff).sum() + lam * (th ** 2).sum()
        p = 1.0 / (1.0 + np.exp(diff))
        g = 2 * lam * th
        np.add.at(g, w, -p); np.add.at(g, l, p)
        return nll, g
    res = minimize(f, np.zeros(len(items)), jac=True, method="L-BFGS-B")
    return pd.Series(res.x, index=items)

def comps_for(g, k):
    return [(r2, r1) if w == 1 else (r1, r2) for r1, r2, w in zip(g.R1, g.R2, g[f"{k}_r2wins"])]

def circular_triads(comps, items):
    s = pd.Series(0, index=items, dtype=float)
    for w_, _ in comps:
        s[w_] += 1
    n = len(items)
    c = n * (n - 1) * (2 * n - 1) / 12 - (s ** 2).sum() / 2
    cmax = (n ** 3 - 4 * n) / 24 if n % 2 == 0 else (n ** 3 - n) / 24
    return c, 1 - c / cmax

ITEMS = {pid: sorted(set(g.R1) | set(g.R2)) for pid, g in long.groupby("post_id")}
CONSTRUCTS = ["cogency", "effect"]
theta, tri = {}, []
for (r, pid), g in long.groupby(["rater", "post_id"]):
    for k in CONSTRUCTS:
        c = comps_for(g, k)
        theta[(r, k, pid)] = fit_bt(ITEMS[pid], c)
        ct, zeta = circular_triads(c, ITEMS[pid])
        tri.append({"rater": r, "construct": k, "post_id": pid, "circular_triads": ct, "consistency_zeta": zeta})
for pid, g in long.groupby("post_id"):
    for k in CONSTRUCTS:
        theta[("POOLED", k, pid)] = fit_bt(ITEMS[pid], comps_for(g, k))
        for r in RATERS:
            theta[(f"OTHERS_{r}", k, pid)] = fit_bt(ITEMS[pid], comps_for(g[g.rater != r], k))

def A(m1, k1, m2, k2):
    """Context-aggregated association: mean within-OP Kendall τ_b."""
    taus = []
    for pid in OP_IDS:
        t1 = theta[(m1, k1, pid)].round(8)
        t2 = theta[(m2, k2, pid)].reindex(t1.index).round(8)
        taus.append(stats.kendalltau(t1.to_numpy(), t2.to_numpy())[0])
    return np.nanmean(taus), taus

rows = []
for k in CONSTRUCTS:
    for r1, r2 in itertools.combinations(RATERS, 2):
        m, t = A(r1, k, r2, k)
        rows.append({"comparison": f"{k}: {r1} vs {r2}", "mean_tau_b": m, **{f"OP{i+1}": x for i, x in enumerate(t)}})
    for r in RATERS:
        m, t = A(r, k, f"OTHERS_{r}", k)
        rows.append({"comparison": f"{k}: {r} vs pooled others", "mean_tau_b": m, **{f"OP{i+1}": x for i, x in enumerate(t)}})
display(Markdown("### Between-rater rank correspondence (same construct)"))
display(pd.DataFrame(rows).set_index("comparison").round(3))

rows = []
for r in RATERS + ["POOLED"]:
    m, t = A(r, "cogency", r, "effect")
    rows.append({"method": r if r != "POOLED" else "Pooled human (all raters)",
                 "A(C, E) mean_tau_b": m, **{f"OP{i+1}": x for i, x in enumerate(t)}})
display(Markdown("### Within-method construct coupling $A(C^H, \\hat E^H)$ — preview of Γ's human component"))
display(pd.DataFrame(rows).set_index("method").round(3))

tri_df = pd.DataFrame(tri)
display(Markdown("### Transitivity: circular triads per rater (max 8 for 6 items; ζ = 1 is perfectly transitive)"))
display(tri_df.pivot_table(index="rater", columns="construct",
                           values=["circular_triads", "consistency_zeta"], aggfunc="mean").round(2))

In [ ]:
# ============================================================
# DISAGREEMENT SHEET (text + every rater's canonical judgment)
# Behavioral outcomes are NOT included.
# ============================================================
base = long[long.rater == RATERS[0]][["pair_key", "post_id", "sample_post_order", "R1", "R2"]].set_index("pair_key")
sheet = base.copy()
for k, lab in (("cogency", "C"), ("effect", "E")):
    W4 = wide(f"{k}_rank4")
    for r in RATERS:
        sheet[f"{lab}_{r}"] = W4[r].map(LABEL4)
    sheet[f"{lab}_unanimous_winner"] = wide(f"{k}_r2wins").nunique(axis=1) == 1
    sheet[f"{lab}_strong_reversal"] = (W4 == 1).any(axis=1) & (W4 == 4).any(axis=1)
Wp = wide("ev_pattern")
for r in RATERS:
    sheet[f"Ev_{r}"] = Wp[r]
sheet["Ev_unanimous_pattern"] = Wp.nunique(axis=1) == 1
for r in RATERS:
    sheet[f"Z_{r}"] = wide("ce_discordant")[r]

sheet["n_disagreements"] = ((~sheet["C_unanimous_winner"]).astype(int) + (~sheet["E_unanimous_winner"]).astype(int)
                            + (~sheet["Ev_unanimous_pattern"]).astype(int))
sheet["op_title"] = sheet["post_id"].map(OP_TITLE)
sheet["R1_text"] = sheet["R1"].map(TEXT)
sheet["R2_text"] = sheet["R2"].map(TEXT)
sheet = sheet.reset_index().sort_values(["sample_post_order", "n_disagreements"], ascending=[True, False])

disagree = sheet[sheet["n_disagreements"] > 0]
print(f"Pairs with any disagreement: {len(disagree)} / {len(sheet)}")
print(f"  Cogency winner split     : {(~sheet.C_unanimous_winner).sum()}  (strong reversals: {sheet.C_strong_reversal.sum()})")
print(f"  Effectiveness winner split: {(~sheet.E_unanimous_winner).sum()}  (strong reversals: {sheet.E_strong_reversal.sum()})")
print(f"  Evidence pattern split   : {(~sheet.Ev_unanimous_pattern).sum()}")

show_cols = ["post_id", "pair_key"] + [c for c in sheet.columns if c[:2] in ("C_", "E_", "Ev")] + ["n_disagreements"]
display(disagree[show_cols].head(40))

In [ ]:
# ============================================================
# PLOTS
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (var, title, w, labs) in zip(axes, [("cogency_r2wins", "Cogency winner κ", None, None),
                                            ("effect_r2wins", "Effectiveness winner κ", None, None),
                                            ("ev_pattern_code", "Evidence pattern κ", None, None)]):
    M = pd.DataFrame(np.eye(len(RATERS)), index=RATERS, columns=RATERS)
    W = wide(var)
    for a, b in itertools.combinations(RATERS, 2):
        M.loc[a, b] = M.loc[b, a] = kappa(W[a], W[b], weights=w, labels=labs)
    im = ax.imshow(M.to_numpy(dtype=float), vmin=-0.2, vmax=1, cmap="viridis")
    ax.set_xticks(range(len(RATERS))); ax.set_xticklabels(RATERS)
    ax.set_yticks(range(len(RATERS))); ax.set_yticklabels(RATERS)
    for i in range(len(RATERS)):
        for j in range(len(RATERS)):
            ax.text(j, i, f"{M.iloc[i, j]:.2f}", ha="center", va="center", color="white", fontsize=11)
    ax.set_title(title)
fig.colorbar(im, ax=axes, shrink=0.8)
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(per_op)); wbar = 0.38
ax.bar(x - wbar / 2, per_op["C_unanimous"], wbar, label="Cogency")
ax.bar(x + wbar / 2, per_op["E_unanimous"], wbar, label="Effectiveness")
ax.axhline(0.25, ls="--", c="gray", lw=1, label="chance unanimity (3 raters, binary)")
ax.set_xticks(x); ax.set_xticklabels([f"OP{i+1}\n{p}" for i, p in enumerate(per_op.post_id)])
ax.set_ylabel("% pairs unanimous on winner"); ax.set_ylim(0, 1); ax.legend()
ax.set_title("Per-OP unanimity")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, k in zip(axes, CONSTRUCTS):
    dist = long.groupby("rater")[f"{k}_rank4"].value_counts(normalize=True).unstack().reindex(columns=[1, 2, 3, 4]).fillna(0)
    dist.columns = [LABEL4[c] for c in dist.columns]
    dist.plot(kind="bar", ax=ax, rot=0)
    ax.set_title(f"{k}: scale use (canonical orientation)"); ax.set_ylabel("share of pairs")
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# SAVE
# ============================================================
summary.to_csv(OUT_DIR / "irr_summary.csv", index=False)
pairwise.to_csv(OUT_DIR / "irr_pairwise.csv", index=False)
pd.DataFrame(loo).to_csv(OUT_DIR / "irr_leave_one_out.csv", index=False)
per_op.to_csv(OUT_DIR / "irr_per_op.csv", index=False)
tend.to_csv(OUT_DIR / "irr_rater_tendencies.csv")
tri_df.to_csv(OUT_DIR / "irr_circular_triads.csv", index=False)
long.to_csv(OUT_DIR / "calibration_long_canonical.csv", index=False)
sheet.to_csv(OUT_DIR / "calibration_disagreement_sheet.csv", index=False, encoding="utf-8-sig")
print("Saved to", OUT_DIR)
for p in sorted(OUT_DIR.glob("*.csv")):
    print("  -", p.name)

## Interpretation notes

- **Primary statistic:** Krippendorff's α (nominal) on the canonical winner, reported separately for cogency and
  expected rhetorical effectiveness (thesis §3.4). Ordinal/interval α on the 4-point scale shows whether
  Strong/Weak use is also shared.
- Common benchmarks (Krippendorff): α ≥ .800 reliable; .667–.800 tentative. Argument-quality and persuasiveness
  judgments are routinely lower, so read the calibration α relative to the subjectivity of the construct.
- **Small sample:** 75 pairs from 5 OPs. The OP-cluster bootstrap resamples only 5 clusters and is therefore crude;
  the unit-level CI is optimistic because pairs within an OP share responses. Treat everything here as diagnostic.
- **Chance unanimity** for 3 raters on a binary choice is 25%.
- **Calibration targets:** strong reversals (Strong R1 vs Strong R2), the evidence-presence threshold, position
  bias (% chose A), and leave-one-out α jumps point to *systematic interpretation differences* — the purpose of
  calibration — rather than to "wrong" answers.
- **C–E discordance and A(C^H, Ê^H):** these preview Study 3/4 quantities; with 5 OPs they are descriptive only.